# Vision Project - Notebook 0

## LabelImg — What It Is, Setup & the Python 3.12+ Fix

**Clause Lens · Data labelling Notebook 1**

This notebook covers:

1. What LabelImg is and what it produces
2. Which Python version to use
3. How to install it on **Windows**, **Linux**, and **macOS**
4. How to set it up in this project (with `uv`)
5. The crash on Python 3.10+ and **the exact code lines changed inside `.venv`**
6. A cell that applies the fix automatically and checks it


## 1. What is LabelImg?

**LabelImg** is a free, open-source desktop tool for drawing **bounding boxes** on images and saving them as
annotation files. You use those files to train object-detection models (YOLO, Faster R-CNN, SSD, …).

| Item                | Detail                                                                         |
| ------------------- | ------------------------------------------------------------------------------ |
| Written in          | Python + **PyQt5** (the GUI toolkit)                                           |
| Latest PyPI version | `labelImg 1.8.6` (no longer updated; the project now points to _Label Studio_) |
| Output formats      | **PascalVOC** (`.xml`), **YOLO** (`.txt`), **CreateML** (`.json`)              |
| Licence             | MIT                                                                            |

### YOLO label format (one `.txt` per image)

```
<class_id> <x_center> <y_center> <width> <height>
```

All values except `class_id` are **normalised to 0–1** by the image width and height. A `classes.txt` file
lists the class names, one per line; the line number is the `class_id`.

### Useful shortcuts

| Key                     | Action                     |
| ----------------------- | -------------------------- |
| `W`                     | Draw a new box             |
| `D` / `A`               | Next / previous image      |
| `Ctrl + S`              | Save                       |
| `Del`                   | Delete the selected box    |
| `Ctrl + Shift + D`      | Delete the current image   |
| `Space`                 | Mark the image as verified |
| `Ctrl + +` / `Ctrl + -` | Zoom in / out              |


## 2. Which Python version?

| Python           | LabelImg 1.8.6 status                                                                                                                                  |
| ---------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **3.6 – 3.9**    | Works with no changes                                                                                                                                  |
| **3.10 – 3.12+** | Opens, then **crashes** when you draw, zoom or scroll (`TypeError: arguments did not match any overloaded call`) unless you apply the fix in Section 5 |

**Why:** Python 3.10 stopped turning `float` values into `int` automatically when they are passed to C
extensions such as PyQt5. LabelImg passes floats (for example `120.5`) to Qt methods that only accept integers
(`drawLine`, `drawRect`, `setValue`, …), so those calls now fail.

**This project** uses **Python 3.12** (see `api/.python-version`) with the fix applied.


## 3. Install on each operating system

> You can install with plain `pip` (shown below) or with `uv` (Section 4). Using a **virtual environment**
> is recommended either way.

### Windows

```powershell
# 1. Install Python 3.9 (no fix needed) or 3.12 (apply the fix) from https://www.python.org
#    Tick "Add Python to PATH" during setup.

# 2. Create and activate a virtual environment
python -m venv .venv
.venv\Scripts\activate

# 3. Install
pip install labelImg

# 4. Run
labelImg
labelImg path\to\images path\to\classes.txt
```

### Linux (Ubuntu / Debian)

```bash
# 1. System packages that Qt needs
sudo apt update
sudo apt install -y python3-venv python3-pip pyqt5-dev-tools libxcb-xinerama0

# 2. Virtual environment
python3 -m venv .venv
source .venv/bin/activate

# 3. Install
pip install labelImg

# 4. Run
labelImg
labelImg path/to/images path/to/classes.txt
```

If you get `qt.qpa.plugin: Could not load the Qt platform plugin "xcb"`, install `libxcb-xinerama0`
(shown above) or run `export QT_QPA_PLATFORM=xcb`.

### macOS (Intel and Apple Silicon)

```bash
# 1. Install Python (Homebrew) — or use uv, which downloads Python for you
brew install python@3.12

# 2. Virtual environment
python3 -m venv .venv
source .venv/bin/activate

# 3. Install (PyQt5 has native wheels for Apple Silicon)
pip install pyqt5 lxml labelImg

# 4. Run
labelImg
labelImg path/to/images path/to/classes.txt
```

The macOS message `WARNING: Secure coding is automatically enabled for restorable state!` is **harmless**;
ignore it. If the window doesn't appear, look for the Python icon in the Dock or press `Cmd + Tab`.


## 4. Setup in this project (`uv`)

The `api/` project already lists LabelImg in `pyproject.toml`:

```toml
dependencies = [
    "labelimg>=1.8.6",
    "pyqt5>=5.15.11",
    "pyqt5-qt5>=5.15.19",
    "setuptools>=84.2.1",   # labelImg imports pkg_resources
]
```

Steps (the same on all three operating systems):

```bash
# 1. Install uv  →  https://docs.astral.sh/uv/getting-started/installation/
#    macOS/Linux:  curl -LsSf https://astral.sh/uv/install.sh | sh
#    Windows:      powershell -c "irm https://astral.sh/uv/install.ps1 | iex"

# 2. From the api/ folder, create .venv and install everything from uv.lock
cd api
uv sync

# 3. Apply the Python 3.12 fix → run the cell in Section 6 of this notebook

# 4. Open LabelImg on our images
uv run labelImg notebooks/raw-data/images
```

Recommended LabelImg settings for this dataset:

1. **Change Save Dir** → `notebooks/raw-data/labels`
2. Toggle the format button under _Save_ to **YOLO**
3. _View → Auto Save mode_ so `D` saves before moving on


## 5. The fix: exact lines changed inside `.venv`

Error seen on Python 3.12:

```
File ".../site-packages/libs/canvas.py", line 530, in paintEvent
    p.drawLine(self.prev_point.x(), 0, self.prev_point.x(), self.pixmap.height())
TypeError: arguments did not match any overloaded call:
  drawLine(self, x1: int, y1: int, x2: int, y2: int): argument 1 has unexpected type 'float'
```

**The fix:** wrap every float passed to an int-only Qt method in `int(...)`.
All paths are relative to `api/.venv/lib/python3.12/site-packages/`
(on Windows: `api\.venv\Lib\site-packages\`).

### 5.1 `libs/canvas.py`

**Line 526**, the box drawn while you drag:

```diff
- p.drawRect(left_top.x(), left_top.y(), rect_width, rect_height)
+ p.drawRect(int(left_top.x()), int(left_top.y()), int(rect_width), int(rect_height))
```

**Lines 530–531**, the crosshair (this caused the crash above):

```diff
- p.drawLine(self.prev_point.x(), 0, self.prev_point.x(), self.pixmap.height())
- p.drawLine(0, self.prev_point.y(), self.pixmap.width(), self.prev_point.y())
+ p.drawLine(int(self.prev_point.x()), 0, int(self.prev_point.x()), int(self.pixmap.height()))
+ p.drawLine(0, int(self.prev_point.y()), int(self.pixmap.width()), int(self.prev_point.y()))
```

### 5.2 `labelImg/labelImg.py`

**Line 965**, scrolling:

```diff
- bar.setValue(bar.value() + bar.singleStep() * units)
+ bar.setValue(int(bar.value() + bar.singleStep() * units))
```

**Line 971**, zoom:

```diff
- self.zoom_widget.setValue(value)
+ self.zoom_widget.setValue(int(value))
```

**Lines 1025–1026**, zooming with the mouse wheel:

```diff
- h_bar.setValue(new_h_bar_value)
- v_bar.setValue(new_v_bar_value)
+ h_bar.setValue(int(new_h_bar_value))
+ v_bar.setValue(int(new_v_bar_value))
```

### 5.3 `libs/shape.py`

**Line 131**, the label text drawn on a box:

```diff
- painter.drawText(min_x, min_y, self.label)
+ painter.drawText(int(min_x), int(min_y), self.label)
```

> These edits live inside `.venv`, which git does not track. Running `uv sync` after deleting `.venv`,
> or reinstalling labelImg, **undoes them**. Re-run the cell in Section 6 each time.


## 6. Apply the fix automatically

Run this cell **with the project's kernel** (`api/.venv`). It:

- finds the installed LabelImg files in the current environment's `site-packages`
- applies each replacement from Section 5
- is **safe to run more than once**; it skips lines that are already fixed


In [1]:
import sys, sysconfig
from pathlib import Path

site = Path(sysconfig.get_paths()["purelib"])
print("Python      :", sys.version.split()[0])
print("site-packages:", site)

PATCHES = {
    "libs/canvas.py": [
        ("p.drawRect(left_top.x(), left_top.y(), rect_width, rect_height)",
         "p.drawRect(int(left_top.x()), int(left_top.y()), int(rect_width), int(rect_height))"),
        ("p.drawLine(self.prev_point.x(), 0, self.prev_point.x(), self.pixmap.height())",
         "p.drawLine(int(self.prev_point.x()), 0, int(self.prev_point.x()), int(self.pixmap.height()))"),
        ("p.drawLine(0, self.prev_point.y(), self.pixmap.width(), self.prev_point.y())",
         "p.drawLine(0, int(self.prev_point.y()), int(self.pixmap.width()), int(self.prev_point.y()))"),
    ],
    "labelImg/labelImg.py": [
        ("bar.setValue(bar.value() + bar.singleStep() * units)",
         "bar.setValue(int(bar.value() + bar.singleStep() * units))"),
        ("self.zoom_widget.setValue(value)",
         "self.zoom_widget.setValue(int(value))"),
        ("h_bar.setValue(new_h_bar_value)", "h_bar.setValue(int(new_h_bar_value))"),
        ("v_bar.setValue(new_v_bar_value)", "v_bar.setValue(int(new_v_bar_value))"),
    ],
    "libs/shape.py": [
        ("painter.drawText(min_x, min_y, self.label)",
         "painter.drawText(int(min_x), int(min_y), self.label)"),
    ],
}

for rel, pairs in PATCHES.items():
    f = site / rel
    if not f.exists():
        print(f"{rel} not found. Is labelImg installed in this kernel?")
        continue
    text = f.read_text(encoding="utf-8")
    for old, new in pairs:
        if new in text:
            print(f"already fixed  {rel}: {new.strip()[:60]}")
        elif old in text:
            text = text.replace(old, new)
            print(f"fixed          {rel}: {new.strip()[:60]}")
        else:
            print(f"not found      {rel}: {old.strip()[:60]}")
    f.write_text(text, encoding="utf-8")

Python      : 3.12.14
site-packages: c:\Abdullah Ahmed\Ai course\Computer Vision\clause-ade\api\.venv\Lib\site-packages
fixed          libs/canvas.py: p.drawRect(int(left_top.x()), int(left_top.y()), int(rect_wi
fixed          libs/canvas.py: p.drawLine(int(self.prev_point.x()), 0, int(self.prev_point.
fixed          libs/canvas.py: p.drawLine(0, int(self.prev_point.y()), int(self.pixmap.widt
fixed          labelImg/labelImg.py: bar.setValue(int(bar.value() + bar.singleStep() * units))
fixed          labelImg/labelImg.py: self.zoom_widget.setValue(int(value))
fixed          labelImg/labelImg.py: h_bar.setValue(int(new_h_bar_value))
fixed          labelImg/labelImg.py: v_bar.setValue(int(new_v_bar_value))
fixed          libs/shape.py: painter.drawText(int(min_x), int(min_y), self.label)


## 7. Verify

This prints every patched line with its line number, and checks that the files still import.


In [2]:
import ast

for rel in PATCHES:
    f = site / rel
    src = f.read_text(encoding="utf-8")
    ast.parse(src)  # raises if a patch broke the syntax
    print(f"\n── {rel}  (syntax OK)")
    for i, line in enumerate(src.splitlines(), 1):
        if any(new in line for _, new in PATCHES[rel]):
            print(f"{i:>5}: {line.strip()}")


── libs/canvas.py  (syntax OK)
  526: p.drawRect(int(left_top.x()), int(left_top.y()), int(rect_width), int(rect_height))
  530: p.drawLine(int(self.prev_point.x()), 0, int(self.prev_point.x()), int(self.pixmap.height()))
  531: p.drawLine(0, int(self.prev_point.y()), int(self.pixmap.width()), int(self.prev_point.y()))

── labelImg/labelImg.py  (syntax OK)
  965: bar.setValue(int(bar.value() + bar.singleStep() * units))
  971: self.zoom_widget.setValue(int(value))
 1025: h_bar.setValue(int(new_h_bar_value))
 1026: v_bar.setValue(int(new_v_bar_value))

── libs/shape.py  (syntax OK)
  131: painter.drawText(int(min_x), int(min_y), self.label)


## 8. Launch and next steps

```bash
cd api
uv run labelImg notebooks/raw-data/images
```

Checklist:

- [ ] Save dir set to `notebooks/raw-data/labels`
- [ ] Format set to **YOLO**
- [ ] `classes.txt` created with our class names
- [ ] Every image in `raw-data/images` has a matching `.txt` in `raw-data/labels`